## 1. Import Libraries

Import the libraries required to load and inspect the competition data. We will use Pandas for CSV files and NumPy for the precomputed feature arrays.

In [37]:
import numpy as np
import pandas as pd
from pathlib import Path

## 2. Dataset Paths

Define the paths to the original competition dataset and its extracted feature files.

The notebook is located inside `ML_PROJECT/Models/Emad`, while the original dataset is stored separately inside `ML_PROJECT/DATA`.

In [38]:
# Project root: ML_PROJECT/
PROJECT_DIR = Path("../../").resolve()

# Original dataset directory
DATA_DIR = PROJECT_DIR / "DATA"
REID_DIR = DATA_DIR / "reid_student_data_main"

# Metadata files
TRAIN_CSV = REID_DIR / "train.csv"
TEST_CSV = REID_DIR / "test.csv"
MIWID_IDS_CSV = REID_DIR / "miewid_ids.csv"
SAMPLE_SUBMISSION_CSV = REID_DIR / "sample_submission.csv"

# MIEW-ID features
MIWID_FEATURES_NPY = REID_DIR / "miewid_features.npy"

# Extracted SIFT directories
SIFT_TRAIN_DIR = DATA_DIR / "sift_features_train.npz"
SIFT_TEST_DIR = DATA_DIR / "sift_features_test.npz"

# Original image directories
HYENA_IMAGES_DIR = DATA_DIR / "train_images_hyena"
LEOPARD_IMAGES_DIR = DATA_DIR / "train_images_leopard"

# Output directory for the unified dataset
OUTPUT_DIR = PROJECT_DIR / "Processed_Data"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MASTER_CSV = OUTPUT_DIR / "master_dataset.csv"

## 3. Load CSV Files

Load the training metadata, test metadata, feature-to-image ID mapping, and sample submission template.

These files contain the image IDs, species, viewpoints, training identities, and the required submission format.

In [39]:
train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)

miewid_ids_df = pd.read_csv(MIWID_IDS_CSV)

sample_submission_df = pd.read_csv(SAMPLE_SUBMISSION_CSV)

## 4. Load MIEW-ID Features

Load the precomputed global image descriptors stored in the NumPy file.

Each row represents an image's feature vector. The corresponding image IDs are stored separately in `miewid_ids.csv`.

In [40]:
miewid_features = np.load(MIWID_FEATURES_NPY)

print("MIEW-ID feature shape:", miewid_features.shape)
print("MIEW-ID feature dtype:", miewid_features.dtype)

MIEW-ID feature shape: (9954, 2152)
MIEW-ID feature dtype: float32


## 5. Load SIFT Features

The SIFT feature archives have been extracted into directories containing individual NumPy files.

Each image has two associated files:
- `*_desc.npy`: SIFT feature descriptors.
- `*_xy.npy`: Keypoint coordinates.

We load these files and organize them by image ID, keeping descriptors and coordinates together.

In [41]:
def load_sift_features(directory):
    sift_data = {}

    descriptor_files = sorted(directory.glob("*_desc.npy"))

    for desc_path in descriptor_files:
        image_id = desc_path.name.replace("_desc.npy", "")
        xy_path = directory / f"{image_id}_xy.npy"

        descriptors = np.load(desc_path)

        if xy_path.exists():
            keypoints = np.load(xy_path)
        else:
            keypoints = None

        sift_data[image_id] = {
            "descriptors": descriptors,
            "keypoints": keypoints
        }

    return sift_data


sift_train = load_sift_features(SIFT_TRAIN_DIR)
sift_test = load_sift_features(SIFT_TEST_DIR)

## 6. Inspect Loaded Metadata

Display the first few rows of the loaded CSV files to confirm that the data has been loaded correctly and that the expected columns are present.

In [42]:
print("Training data:")
display(train_df.head())

print("\nTest data:")
display(test_df.head())

print("\nMIEW-ID image ID mapping:")
display(miewid_ids_df.head())

print("\nSample submission:")
display(sample_submission_df.head())

Training data:


,id,species,viewpoint,identity
0,tr_00001,hyena,left,HYE0079
1,tr_00002,leopard,back,LEO0017
2,tr_00003,leopard,left,LEO0575
3,tr_00004,hyena,left,HYE0222
4,tr_00005,leopard,right,LEO0412



Test data:


,id,species,viewpoint
0,te_00001,leopard,right
1,te_00002,hyena,left
2,te_00003,leopard,right
3,te_00004,leopard,left
4,te_00005,leopard,right



MIEW-ID image ID mapping:


,id
0,tr_00001
1,tr_00002
2,tr_00003
3,tr_00004
4,tr_00005



Sample submission:


,id,predictions
0,te_00001,new_individual
1,te_00002,new_individual
2,te_00003,new_individual
3,te_00004,new_individual
4,te_00005,new_individual


In [43]:
print("========== MIEW-ID FEATURES ==========")
print("Shape:", miewid_features.shape)
print("Data type:", miewid_features.dtype)
print("Number of mapped IDs:", len(miewid_ids_df))

print("\n========== SIFT FEATURES ==========")
print("Training images:", len(sift_train))
print("Test images:", len(sift_test))

# Inspect one example from each SIFT dataset
for name, data in [("Train", sift_train), ("Test", sift_test)]:
    sample_id = next(iter(data))
    print(f"\n{name} example: {sample_id}")
    print("Descriptors:", data[sample_id]["descriptors"].shape)
    print("Keypoints:", data[sample_id]["keypoints"].shape)

========== MIEW-ID FEATURES ==========
Shape: (9954, 2152)
Data type: float32
Number of mapped IDs: 9954

========== SIFT FEATURES ==========
Training images: 7163
Test images: 2791

Train example: tr_00001
Descriptors: (500, 128)
Keypoints: (500, 2)

Test example: te_00001
Descriptors: (500, 128)
Keypoints: (500, 2)


## 7. Build Unified Metadata

Combine the training and test metadata into a single table.

Each row represents one image. The `split` column identifies whether the image belongs to the training or test set, allowing both datasets to be handled through a common interface.

In [44]:
# Add split information
train_metadata = train_df.copy()
test_metadata = test_df.copy()

train_metadata["split"] = "train"
test_metadata["split"] = "test"

# Combine both datasets
master_df = pd.concat(
    [train_metadata, test_metadata],
    ignore_index=True
)

# Ensure missing test identities remain empty
master_df["identity"] = master_df["identity"].where(
    master_df["split"] == "train",
    np.nan
)

print("Total images:", len(master_df))
print("Training images:", (master_df["split"] == "train").sum())
print("Test images:", (master_df["split"] == "test").sum())

display(master_df.head())

Total images: 9954
Training images: 7163
Test images: 2791


,id,species,viewpoint,identity,split
0,tr_00001,hyena,left,HYE0079,train
1,tr_00002,leopard,back,LEO0017,train
2,tr_00003,leopard,left,LEO0575,train
3,tr_00004,hyena,left,HYE0222,train
4,tr_00005,leopard,right,LEO0412,train


## 8. Map MIEW-ID Features

Associate every image with its corresponding row in the MIEW-ID feature matrix.

The master dataset will store the feature matrix path and the row index required to retrieve an image's descriptor.

In [45]:
# Check that the ID mapping and feature matrix have matching lengths
assert len(miewid_ids_df) == len(miewid_features), \
    "MIEW-ID mapping and feature matrix have different lengths."

assert miewid_ids_df["id"].is_unique, \
    "Duplicate image IDs found in MIEW-ID mapping."

# Create image ID -> feature row index mapping
miewid_index_map = {
    image_id: index
    for index, image_id in enumerate(miewid_ids_df["id"])
}

# Add mappings to master dataset
master_df["miewid_index"] = master_df["id"].map(miewid_index_map)

# Store the feature matrix path relative to the project root
master_df["miewid_features_path"] = str(
    MIWID_FEATURES_NPY.relative_to(PROJECT_DIR)
)

print("Images mapped:", master_df["miewid_index"].notna().sum())
print("Images without MIEW-ID mapping:", master_df["miewid_index"].isna().sum())

Images mapped: 9954
Images without MIEW-ID mapping: 0


## 9. Map SIFT Features

Associate every image with its SIFT descriptor and keypoint files.

The extracted SIFT directories contain separate NumPy files for descriptors and keypoint coordinates. We map these files using their image IDs and store their paths in the master dataset.

In [46]:
def build_sift_mapping(directory):
    mapping = {}

    for desc_path in directory.glob("*_desc.npy"):
        image_id = desc_path.name.removesuffix("_desc.npy")
        xy_path = directory / f"{image_id}_xy.npy"

        mapping[image_id] = {
            "sift_desc_path": desc_path,
            "sift_xy_path": xy_path if xy_path.exists() else None
        }

    return mapping


# Build mappings for both splits
sift_mapping = {}

sift_mapping.update(build_sift_mapping(SIFT_TRAIN_DIR))
sift_mapping.update(build_sift_mapping(SIFT_TEST_DIR))

# Attach paths to master dataset
master_df["sift_desc_path"] = master_df["id"].map(
    lambda image_id: (
        str(sift_mapping[image_id]["sift_desc_path"].relative_to(PROJECT_DIR))
        if image_id in sift_mapping else np.nan
    )
)

master_df["sift_xy_path"] = master_df["id"].map(
    lambda image_id: (
        str(sift_mapping[image_id]["sift_xy_path"].relative_to(PROJECT_DIR))
        if image_id in sift_mapping
        and sift_mapping[image_id]["sift_xy_path"] is not None
        else np.nan
    )
)

print("SIFT images mapped:", master_df["sift_desc_path"].notna().sum())
print("Missing descriptors:", master_df["sift_desc_path"].isna().sum())
print("Missing keypoints:", master_df["sift_xy_path"].isna().sum())

SIFT images mapped: 9954
Missing descriptors: 0
Missing keypoints: 0


## 10. Validate Dataset Mappings

Perform consistency checks on the unified dataset.

Verify that:
- Every image has a unique ID.
- All images have a corresponding MIEW-ID feature vector.
- All images have SIFT descriptors and keypoint coordinates.
- The training and test splits contain the expected number of images.

These checks ensure that the master dataset can be used reliably by subsequent models.

In [47]:
# Expected dataset size
assert len(master_df) == 9954
assert master_df["id"].is_unique

# MIEW-ID validation
assert master_df["miewid_index"].notna().all()

# SIFT validation
assert master_df["sift_desc_path"].notna().all()
assert master_df["sift_xy_path"].notna().all()

# Verify every referenced SIFT file exists
for column in ["sift_desc_path", "sift_xy_path"]:
    missing_files = [
        path for path in master_df[column]
        if not (PROJECT_DIR / path).is_file()
    ]

    assert len(missing_files) == 0, \
        f"Missing files found in {column}: {missing_files[:5]}"

# Convert feature indices to integers after confirming no missing mappings
master_df["miewid_index"] = master_df["miewid_index"].astype(int)

# Split validation
assert (master_df["split"] == "train").sum() == 7163
assert (master_df["split"] == "test").sum() == 2791

print("All validation checks passed!")

All validation checks passed!


## 11. Save Unified Dataset

Save the completed master dataset as a CSV file.

Feature arrays remain in their original NumPy files. The CSV stores their paths and corresponding indices, allowing future models to retrieve all required data through a single master dataset.

In [48]:
master_df.to_csv(MASTER_CSV, index=False)

print("Master dataset saved successfully!")
print("Location:", MASTER_CSV)
print("Shape:", master_df.shape)

display(master_df.head())

Master dataset saved successfully!
Location: /media/emadtariqu/Storage/Linux/ML_PROJECT/Processed_Data/master_dataset.csv
Shape: (9954, 9)


,id,species,viewpoint,identity,split,miewid_index,miewid_features_path,sift_desc_path,sift_xy_path
0,tr_00001,hyena,left,HYE0079,train,0,DATA/reid_student_data_main/miewid_features.npy,DATA/sift_features_train.npz/tr_00001_desc.npy,DATA/sift_features_train.npz/tr_00001_xy.npy
1,tr_00002,leopard,back,LEO0017,train,1,DATA/reid_student_data_main/miewid_features.npy,DATA/sift_features_train.npz/tr_00002_desc.npy,DATA/sift_features_train.npz/tr_00002_xy.npy
2,tr_00003,leopard,left,LEO0575,train,2,DATA/reid_student_data_main/miewid_features.npy,DATA/sift_features_train.npz/tr_00003_desc.npy,DATA/sift_features_train.npz/tr_00003_xy.npy
3,tr_00004,hyena,left,HYE0222,train,3,DATA/reid_student_data_main/miewid_features.npy,DATA/sift_features_train.npz/tr_00004_desc.npy,DATA/sift_features_train.npz/tr_00004_xy.npy
4,tr_00005,leopard,right,LEO0412,train,4,DATA/reid_student_data_main/miewid_features.npy,DATA/sift_features_train.npz/tr_00005_desc.npy,DATA/sift_features_train.npz/tr_00005_xy.npy


## 12. Package MIEW-ID Feature Vectors

Copy the complete MIEW-ID feature matrix into the processed dataset directory.

The rows retain their original ordering, and `miewid_index` in the master dataset identifies the corresponding feature vector for each image.

In [49]:
import shutil

PROCESSED_MIWEID = OUTPUT_DIR / "miewid_features.npy"

shutil.copy2(MIWID_FEATURES_NPY, PROCESSED_MIWEID)

# Update the master dataset to reference the packaged feature matrix
master_df["miewid_features_path"] = "miewid_features.npy"

print("MIEW-ID features packaged.")
print("Shape:", np.load(PROCESSED_MIWEID, mmap_mode="r").shape)
print("Size (MB):", round(PROCESSED_MIWEID.stat().st_size / (1024 ** 2), 2))

MIEW-ID features packaged.
Shape: (9954, 2152)
Size (MB): 81.71


## 13. Package SIFT Feature Vectors

Combine the individual SIFT descriptor and keypoint NumPy files into one compressed archive.

Each image is identified by its unique image ID. This allows future models to retrieve SIFT features directly without navigating individual files.

In [50]:
sift_arrays = {}

for image_id, paths in sift_mapping.items():
    desc_path = paths["sift_desc_path"]
    xy_path = paths["sift_xy_path"]

    if xy_path is None:
        raise FileNotFoundError(f"Missing keypoints for {image_id}")

    sift_arrays[f"{image_id}_desc"] = np.load(desc_path)
    sift_arrays[f"{image_id}_xy"] = np.load(xy_path)

SIFT_BUNDLE = OUTPUT_DIR / "sift_features.npz"

np.savez_compressed(SIFT_BUNDLE, **sift_arrays)

print("SIFT features packaged.")
print("Images:", len(sift_arrays) // 2)
print("Arrays:", len(sift_arrays))
print("Size (MB):", round(SIFT_BUNDLE.stat().st_size / (1024 ** 2), 2))

SIFT features packaged.
Images: 9954
Arrays: 19908
Size (MB): 457.53


## 14. Finalize Unified Dataset

Update the master dataset to reference the packaged feature files.

The master CSV now provides all metadata and the indices required to retrieve the actual feature vectors.

In [51]:
# Replace individual SIFT file paths with the common archive reference
master_df = master_df.drop(
    columns=["sift_desc_path", "sift_xy_path"]
)

master_df["sift_features_path"] = "sift_features.npz"

# Save the updated master dataset
master_df.to_csv(MASTER_CSV, index=False)

print("Final master dataset saved.")
print("Shape:", master_df.shape)
display(master_df.head())

Final master dataset saved.
Shape: (9954, 8)


,id,species,viewpoint,identity,split,miewid_index,miewid_features_path,sift_features_path
0,tr_00001,hyena,left,HYE0079,train,0,miewid_features.npy,sift_features.npz
1,tr_00002,leopard,back,LEO0017,train,1,miewid_features.npy,sift_features.npz
2,tr_00003,leopard,left,LEO0575,train,2,miewid_features.npy,sift_features.npz
3,tr_00004,hyena,left,HYE0222,train,3,miewid_features.npy,sift_features.npz
4,tr_00005,leopard,right,LEO0412,train,4,miewid_features.npy,sift_features.npz


## 15. Test Feature Retrieval

Load the packaged feature files and retrieve the MIEW-ID and SIFT features for a sample image.

This verifies that the master dataset correctly links image metadata to its numerical feature representations.

In [52]:
# Load the packaged feature data
miewid = np.load(OUTPUT_DIR / "miewid_features.npy", mmap_mode="r")
sift = np.load(OUTPUT_DIR / "sift_features.npz")

# Select one image
sample = master_df.iloc[0]
image_id = sample["id"]

# Retrieve MIEW-ID vector
miewid_vector = miewid[int(sample["miewid_index"])]

# Retrieve SIFT features
sift_descriptors = sift[f"{image_id}_desc"]
sift_keypoints = sift[f"{image_id}_xy"]

print("Image:", image_id)
print("MIEW-ID vector:", miewid_vector.shape)
print("SIFT descriptors:", sift_descriptors.shape)
print("SIFT keypoints:", sift_keypoints.shape)

Image: tr_00001
MIEW-ID vector: (2152,)
SIFT descriptors: (500, 128)
SIFT keypoints: (500, 2)


# **Actual Model Implementation**
### 1. Use MIEW-ID features only. No SIFT yet.
### 2. Compare images using cosine similarity.
### 3. Restrict comparisons to the same species.
### 4. Aggregate the nearest images into ranked identity predictions.
### 5. Evaluate using MAP@5 on a validation split before generating a Kaggle submission.

In [54]:
# Separate training metadata
model_df = master_df[master_df["split"] == "train"].copy()

# Reproducible validation split
rng = np.random.default_rng(42)

train_indices = []
val_indices = []

for identity, group in model_df.groupby("identity"):
    indices = group.index.to_numpy().copy()
    rng.shuffle(indices)

    if len(indices) >= 2:
        n_val = max(1, int(round(len(indices) * 0.2)))
        n_val = min(n_val, len(indices) - 1)

        val_indices.extend(indices[:n_val])
        train_indices.extend(indices[n_val:])
    else:
        # Keep identities with only one image in training
        train_indices.extend(indices)

model_train_df = model_df.loc[train_indices].reset_index(drop=True)
model_val_df = model_df.loc[val_indices].reset_index(drop=True)

print("Training images:", len(model_train_df))
print("Validation images:", len(model_val_df))
print("Training identities:", model_train_df["identity"].nunique())
print("Validation identities:", model_val_df["identity"].nunique())

# Verify every validation identity exists in training
assert set(model_val_df["identity"]).issubset(
    set(model_train_df["identity"])
)

print("Validation split created successfully!")

Training images: 5687
Validation images: 1476
Training identities: 550
Validation identities: 448
Validation split created successfully!


In [56]:
# Load the packaged MIEW-ID feature matrix
miewid_matrix = np.load(
    OUTPUT_DIR / "miewid_features.npy",
    mmap_mode="r"
)

# Get the correct feature vectors using the master dataset indices
train_feature_indices = model_train_df["miewid_index"].to_numpy(dtype=int)
val_feature_indices = model_val_df["miewid_index"].to_numpy(dtype=int)

train_vectors = miewid_matrix[train_feature_indices]
val_vectors = miewid_matrix[val_feature_indices]

# L2-normalize the vectors
train_norms = np.linalg.norm(train_vectors, axis=1, keepdims=True)
val_norms = np.linalg.norm(val_vectors, axis=1, keepdims=True)

train_vectors = train_vectors / np.maximum(train_norms, 1e-12)
val_vectors = val_vectors / np.maximum(val_norms, 1e-12)

print("Training feature matrix:", train_vectors.shape)
print("Validation feature matrix:", val_vectors.shape)

print("Example training vector norm:", np.linalg.norm(train_vectors[0]))
print("Example validation vector norm:", np.linalg.norm(val_vectors[0]))

# Calculate cosine similarities between validation and training images
similarity_matrix = val_vectors @ train_vectors.T

print("Similarity matrix shape:", similarity_matrix.shape)
print("Minimum similarity:", similarity_matrix.min())
print("Maximum similarity:", similarity_matrix.max())
print("Mean similarity:", similarity_matrix.mean())

Training feature matrix: (5687, 2152)
Validation feature matrix: (1476, 2152)
Example training vector norm: 0.99999994
Example validation vector norm: 1.0
Similarity matrix shape: (1476, 5687)
Minimum similarity: -0.13457377
Maximum similarity: 0.99999976
Mean similarity: 0.16367197


In [57]:
# Metadata aligned with the feature matrices
train_species = model_train_df["species"].to_numpy()
train_identities = model_train_df["identity"].to_numpy()

val_species = model_val_df["species"].to_numpy()
val_identities = model_val_df["identity"].to_numpy()

predictions = []

for i in range(len(model_val_df)):

    # Only compare images belonging to the same species
    species_mask = train_species == val_species[i]

    similarities = similarity_matrix[i, species_mask]
    candidate_identities = train_identities[species_mask]

    # Aggregate image similarities by identity, taking the maximum
    unique_ids, inverse_indices = np.unique(
        candidate_identities,
        return_inverse=True
    )

    identity_scores = np.full(len(unique_ids), -np.inf)
    np.maximum.at(identity_scores, inverse_indices, similarities)

    # Rank identities by descending similarity
    ranked_indices = np.argsort(identity_scores)[::-1]

    top_5 = unique_ids[ranked_indices[:5]]
    predictions.append(top_5.tolist())

print("Number of predictions:", len(predictions))
print("Example predictions:", predictions[0])
print("Actual identity:", val_identities[0])

Number of predictions: 1476
Example predictions: ['HYE0056', 'HYE0055', 'HYE0000', 'HYE0180', 'HYE0118']
Actual identity: HYE0000


In [58]:
def map_at_5(y_true, y_pred):
    scores = []

    for actual, predicted in zip(y_true, y_pred):
        score = 0.0

        for rank, identity in enumerate(predicted[:5], start=1):
            if identity == actual:
                score = 1.0 / rank
                break

        scores.append(score)

    return np.mean(scores)


baseline_map5 = map_at_5(val_identities, predictions)

print(f"Model 1 — MIEW-ID Cosine Similarity")
print(f"Validation MAP@5: {baseline_map5:.5f}")
print(f"Validation MAP@5 (%): {baseline_map5 * 100:.2f}%")

Model 1 — MIEW-ID Cosine Similarity
Validation MAP@5: 0.85638
Validation MAP@5 (%): 85.64%


In [59]:
top1 = np.mean([
    actual in predicted[:1]
    for actual, predicted in zip(val_identities, predictions)
])

top3 = np.mean([
    actual in predicted[:3]
    for actual, predicted in zip(val_identities, predictions)
])

top5 = np.mean([
    actual in predicted[:5]
    for actual, predicted in zip(val_identities, predictions)
])

print(f"Top-1 Accuracy: {top1 * 100:.2f}%")
print(f"Top-3 Accuracy: {top3 * 100:.2f}%")
print(f"Top-5 Accuracy: {top5 * 100:.2f}%")

Top-1 Accuracy: 83.06%
Top-3 Accuracy: 88.14%
Top-5 Accuracy: 89.57%


In [60]:
# Evaluate performance separately for each species
for species in model_val_df["species"].unique():

    mask = val_species == species

    species_map5 = map_at_5(
        val_identities[mask],
        [predictions[i] for i in np.where(mask)[0]]
    )

    species_top1 = np.mean([
        val_identities[i] == predictions[i][0]
        for i in np.where(mask)[0]
    ])

    species_top5 = np.mean([
        val_identities[i] in predictions[i][:5]
        for i in np.where(mask)[0]
    ])

    print(f"\nSpecies: {species}")
    print(f"Validation images: {mask.sum()}")
    print(f"MAP@5: {species_map5 * 100:.2f}%")
    print(f"Top-1: {species_top1 * 100:.2f}%")
    print(f"Top-5: {species_top5 * 100:.2f}%")


Species: hyena
Validation images: 396
MAP@5: 85.11%
Top-1: 83.08%
Top-5: 88.38%

Species: leopard
Validation images: 1080
MAP@5: 85.83%
Top-1: 83.06%
Top-5: 90.00%


In [61]:
# Collect incorrectly ranked predictions
wrong_predictions = []

for i in range(len(val_identities)):
    if predictions[i][0] != val_identities[i]:
        wrong_predictions.append({
            "id": model_val_df.iloc[i]["id"],
            "species": val_species[i],
            "actual": val_identities[i],
            "predicted": predictions[i][0],
            "top_5": predictions[i]
        })

wrong_df = pd.DataFrame(wrong_predictions)

print("Incorrect Top-1 predictions:", len(wrong_df))
display(wrong_df.head(15))

Incorrect Top-1 predictions: 250


,id,species,actual,predicted,top_5
0,tr_04539,hyena,HYE0000,HYE0056,"[HYE0056, HYE0055, HYE0000, HYE0180, HYE0118]"
1,tr_03340,hyena,HYE0020,HYE0446,"[HYE0446, HYE0058, HYE0590, HYE0020, HYE0275]"
2,tr_00570,hyena,HYE0031,HYE0648,"[HYE0648, HYE0524, HYE0031, HYE0400, HYE0398]"
3,tr_02187,hyena,HYE0034,HYE0511,"[HYE0511, HYE0513, HYE0034, HYE0448, HYE0092]"
4,tr_00639,hyena,HYE0034,HYE0446,"[HYE0446, HYE0513, HYE0177, HYE0433, HYE0390]"
5,tr_03126,hyena,HYE0034,HYE0446,"[HYE0446, HYE0513, HYE0177, HYE0390, HYE0275]"
6,tr_06882,hyena,HYE0041,HYE0583,"[HYE0583, HYE0513, HYE0034, HYE0346, HYE0011]"
7,tr_05973,hyena,HYE0054,HYE0560,"[HYE0560, HYE0054, HYE0013, HYE0420, HYE0079]"
8,tr_06998,hyena,HYE0061,HYE0111,"[HYE0111, HYE0566, HYE0283, HYE0241, HYE0210]"
9,tr_01882,hyena,HYE0067,HYE0013,"[HYE0013, HYE0515, HYE0126, HYE0236, HYE0207]"


## Testing effect of viewpoints

In [63]:
print("Training viewpoint distribution:")
display(
    model_train_df.groupby(["species", "viewpoint"])
    .size()
    .unstack(fill_value=0)
)

print("\nValidation viewpoint distribution:")
display(
    model_val_df.groupby(["species", "viewpoint"])
    .size()
    .unstack(fill_value=0)
)

print("\nMissing viewpoints:")
print("Train:", model_train_df["viewpoint"].isna().sum())
print("Validation:", model_val_df["viewpoint"].isna().sum())

Training viewpoint distribution:


viewpoint,back,backleft,backright,down,front,frontleft,frontright,left,right,up
species,,,,,,,,,,
hyena,14,36,40,5,28,70,46,624,631,22
leopard,701,0,0,146,582,0,0,1313,1321,0



Validation viewpoint distribution:


viewpoint,back,backleft,backright,down,front,frontleft,frontright,left,right,up
species,,,,,,,,,,
hyena,2,8,13,3,10,19,6,151,178,6
leopard,169,0,0,36,139,0,0,380,328,0



Missing viewpoints:
Train: 108
Validation: 28


In [64]:
train_viewpoints = model_train_df["viewpoint"].to_numpy()
val_viewpoints = model_val_df["viewpoint"].to_numpy()

viewpoint_coverage = []

for i in range(len(model_val_df)):

    # Training images of the same species and viewpoint
    mask = (
        (train_species == val_species[i]) &
        (train_viewpoints == val_viewpoints[i])
    )

    candidate_ids = train_identities[mask]

    viewpoint_coverage.append({
        "species": val_species[i],
        "viewpoint": val_viewpoints[i],
        "has_matching_viewpoint": pd.notna(val_viewpoints[i]),
        "has_correct_identity": val_identities[i] in candidate_ids,
        "candidate_count": len(candidate_ids)
    })

coverage_df = pd.DataFrame(viewpoint_coverage)

print("Overall:")
print(
    "Correct identity available:",
    f"{coverage_df['has_correct_identity'].mean() * 100:.2f}%"
)

print("\nBy species:")
display(
    coverage_df.groupby("species").agg(
        validation_images=("species", "size"),
        correct_identity_available=("has_correct_identity", "mean"),
        average_candidates=("candidate_count", "mean")
    )
)

Overall:
Correct identity available: 89.09%

By species:


,validation_images,correct_identity_available,average_candidates
species,,,
hyena,396,0.866162,528.815657
leopard,1080,0.900000,1052.639815


### Controlled weighted score of viewpoint

In [65]:
viewpoint_bonus_results = {}

for bonus in [0, 0.01, 0.02, 0.05, 0.10]:

    bonus_predictions = []

    for i in range(len(model_val_df)):

        species_mask = train_species == val_species[i]

        similarities = similarity_matrix[i, species_mask].copy()
        candidate_ids = train_identities[species_mask]
        candidate_viewpoints = train_viewpoints[species_mask]

        # Add a bonus for matching, non-missing viewpoints
        if pd.notna(val_viewpoints[i]):
            matching = (
                candidate_viewpoints == val_viewpoints[i]
            ) & pd.notna(candidate_viewpoints)

            similarities[matching] += bonus

        bonus_predictions.append(
            rank_identities(similarities, candidate_ids, "max")
        )

    score = map_at_5(val_identities, bonus_predictions)
    viewpoint_bonus_results[bonus] = score

    print(f"Bonus {bonus:.2f}: MAP@5 = {score:.5f}")

Bonus 0.00: MAP@5 = 0.85638
Bonus 0.01: MAP@5 = 0.85610
Bonus 0.02: MAP@5 = 0.85470
Bonus 0.05: MAP@5 = 0.85253
Bonus 0.10: MAP@5 = 0.84648


### Above didnt work out, so we are defaulting to original idea

In [66]:
print("Sample submission:")
display(sample_submission_df.head(10))

print("\nData types:")
print(sample_submission_df.dtypes)

print("\nNumber of rows:", len(sample_submission_df))

Sample submission:


,id,predictions
0,te_00001,new_individual
1,te_00002,new_individual
2,te_00003,new_individual
3,te_00004,new_individual
4,te_00005,new_individual
5,te_00006,new_individual
6,te_00007,new_individual
7,te_00008,new_individual
8,te_00009,new_individual
9,te_00010,new_individual



Data types:
id             str
predictions    str
dtype: object

Number of rows: 2791


In [67]:
# Calculate similarities against ALL training images
test_similarity_matrix = test_vectors @ full_train_vectors.T

# Training metadata
full_train_species = full_train_df["species"].to_numpy()
full_train_identities = full_train_df["identity"].to_numpy()

test_species = test_model_df["species"].to_numpy()

test_predictions = []

for i in range(len(test_model_df)):

    # Restrict candidates to the same species
    species_mask = full_train_species == test_species[i]

    similarities = test_similarity_matrix[i, species_mask]
    candidate_ids = full_train_identities[species_mask]

    # Rank identities using maximum similarity
    unique_ids, inverse_indices = np.unique(
        candidate_ids,
        return_inverse=True
    )

    identity_scores = np.full(len(unique_ids), -np.inf)
    np.maximum.at(identity_scores, inverse_indices, similarities)

    ranked_indices = np.argsort(identity_scores)[::-1]
    top_5 = unique_ids[ranked_indices[:5]]

    test_predictions.append(" ".join(top_5))

print("Test predictions generated:", len(test_predictions))
print("Example:", test_predictions[0])

NameError: name 'test_vectors' is not defined